<div align="center">
  <h1>LLM Evaluation with LangSmith</h1>
  <h3>Benchmarking summarization quality using embedding distance metrics</h3>
  <p>by <b><a href="https://www.linkedin.com/in/adebanji-adelowo/">Adebanji Oluwatimileyin Adelowo</a></b> &nbsp;|&nbsp; <a href="https://github.com/AdebanjiAdelowo">GitHub</a></p>
</div>

---

**Models:** Flan-T5-base (zero-shot) | T5-CNN-DM (fine-tuned) | OpenAI GPT-4o-mini  
**Environment:** Google Colab (CPU) or local  
**Tags:** LangSmith, Evaluation, Embeddings, Summarization, T5, OpenAI

---

In [72]:
!pip install -q --upgrade langchain langchain-core langchain-openai langchain-community langchain-huggingface langsmith datasets huggingface-hub transformers rapidfuzz openai tiktoken python-dotenv

In [73]:
!pip install rouge_score

In [74]:
#You need a LangChain API Key.
from getpass import getpass
import os
if not 'LANGCHAIN_API_KEY' in os.environ:
  os.environ["LANGCHAIN_API_KEY"] = getpass("LangChain API Key: ")

In [75]:
if not 'OPENAI_API_KEY' in os.environ:
  os.environ["OPENAI_API_KEY"] = getpass("OPENAI API Key: ")

In [76]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"]="https://api.smith.langchain.com"
#os.environ["LANGCHAIN_PROJECT"]="langsmith_yttest01"

In [77]:
#Importing Client from Langsmith
from langsmith import Client
client = Client()

## Create Dataset


In [78]:
from datasets import load_dataset

cnn_dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

In [79]:
#Get just a few news to test
MAX_NEWS=3
sample_cnn = cnn_dataset["test"].select(range(MAX_NEWS))

sample_cnn

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 3
})

The dataset contains three columns: article, highlights, and id. To use LangSmith, we need to create a dataset in LangSmith format.

LangSmith expects a prompt and a result. To achieve this, we will transform the article into a prompt by adding the prefix: "Summarize this news." As a result, we will use the content of highlights, which represents the summaries created by humans.

In [80]:
print(sample_cnn[0])

{'article': '(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC\'s founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians\' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednesday

Now We have the Dataset with the prompt and the Reference Summary, it is time to create a Dataset in LangSmith with this information.
### Create the Dataset in Langsmith

The dataset in LangSmith is composed of an input, which is the prompt passed to the model for evaluation, and an output, which should contain what we expect the model to return.

In [81]:
#import uuid
import datetime
input_key=['article']
output_key=['highlights']

NAME_DATASET=f"Summarize_dataset_{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

In [82]:
dataset = client.upload_dataframe(
    df=sample_cnn.to_pandas(),
    input_keys=input_key,
    output_keys=output_key,
    name=NAME_DATASET,
    description="Test Embedding distance between model summarizations",
    data_type="kv"
)

In this image, we can see an example from the dataset once it's been registered in LangSmith.

In the Input column, there is the prompt to be sent, while in the Output column, the expected output is stored.

When performing the comparison, the model will be given the prompt, and the Cosine distance between its response and the one stored in the sample dataset will be calculated.
<img src="https://github.com/AdebanjiAdelowo/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Dataset.jpg?raw=true">

### Recovering Models From Hugging Face
Let's retrieve both models from HuggingFace. A base T5 model and a model that has been fine-tuned using the training portion of this same dataset to generate summaries.

In [83]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device) 

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [84]:
def summarize(text, model, tokenizer):
    prompt = f"Summarize the following text:\n\n{text}"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=180
    ).to(model.device)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [85]:
def summarizer(inputs):
    return summarize(inputs["article"], model, tokenizer)

In [86]:
model_name_finetuned = "flax-community/t5-base-cnn-dm"

tokenizer_finetuned = AutoTokenizer.from_pretrained(model_name_finetuned)
model_finetuned = AutoModelForSeq2SeqLM.from_pretrained(model_name_finetuned)
model_finetuned.to(device) 

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

T5ForConditionalGeneration(
  (shared): Embedding(32100, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32100, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [87]:
def summarizer_finetuned(inputs):
    return summarize(inputs["article"], model_finetuned, tokenizer_finetuned)

## Defining Evaluator
The first step is to define an evaluator, where we specify the variables we want to evaluate. In our case, I have chosen to measure only the "embedding_distance."

I've left the "string_distance" as a comment in case you want to conduct a test with two evaluations instead of one.


In [88]:
import numpy as np
from langsmith import evaluate
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

def embedding_distance(outputs: dict, reference_outputs: dict) -> dict:
    prediction = outputs.get("output", "")
    reference = reference_outputs.get("highlights", "")
    emb1 = np.array(embeddings_model.embed_query(prediction))
    emb2 = np.array(embeddings_model.embed_query(reference))
    cosine_sim = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return {"key": "embedding_distance", "score": round(1 - cosine_sim, 4)}

In [89]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def rouge_metric(outputs, reference_outputs):
    pred = outputs["output"]
    ref = reference_outputs["highlights"]

    scores = scorer.score(ref, pred)

    return {
        "key": "rougeL",
        "score": scores["rougeL"].fmeasure
    }

In [90]:
evaluators=[embedding_distance, rouge_metric]

## Running Evaluator
With the same configuration, we can launch two evaluations on the same dataset. One for each of the chosen models.

In [91]:
base_t5_results = evaluate(
    lambda inputs: {"output": summarizer(inputs)},
    data=NAME_DATASET,
    evaluators=evaluators,
    experiment_prefix="T5-BASE",
    client=client,
)

View the evaluation results for experiment: 'T5-BASE-ed6d51ad' at:
https://smith.langchain.com/o/e84aac92-db27-4e0f-a074-ce87a4032464/datasets/d2b06385-876c-44ab-b0a8-601461c61060/compare?selectedSessions=3ea2f4d5-8078-472e-b4aa-17e1892d3dd5




0it [00:00, ?it/s]

In [92]:
finetuned_t5_results = evaluate(
    lambda inputs: {"output": summarizer_finetuned(inputs)},
    data=NAME_DATASET,
    evaluators=evaluators,
    experiment_prefix="T5-FineTuned",
    client=client,
)

View the evaluation results for experiment: 'T5-FineTuned-69778e93' at:
https://smith.langchain.com/o/e84aac92-db27-4e0f-a074-ce87a4032464/datasets/d2b06385-876c-44ab-b0a8-601461c61060/compare?selectedSessions=a2e11dbc-0827-4ce2-a316-94f23c66b5ce




0it [00:00, ?it/s]

<img src="https://github.com/AdebanjiAdelowo/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Tests.jpg?raw=true">

In the image below you can see the comparision between two tests.
<img src="https://github.com/AdebanjiAdelowo/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareTestst.jpg?raw=true">

Well, since it has been so straightforward, why don't we try to make the comparison with an OpenAI model?

In [93]:
if not 'OPENAI_API_KEY' in os.environ:
  os.environ["OPENAI_API_KEY"] = getpass("OPENAI API Key: ")

In [94]:
from langchain_openai import ChatOpenAI

openai_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [95]:
openai_results = evaluate(
    lambda inputs: {"output": openai_llm.invoke(inputs["article"]).content},
    data=NAME_DATASET,
    evaluators=evaluators,
    experiment_prefix="OpenAI-GPT4o-mini",
    client=client,
)

View the evaluation results for experiment: 'OpenAI-GPT4o-mini-5edee1f6' at:
https://smith.langchain.com/o/e84aac92-db27-4e0f-a074-ce87a4032464/datasets/d2b06385-876c-44ab-b0a8-601461c61060/compare?selectedSessions=49225d12-c929-4716-bb55-24fe74e32430




0it [00:00, ?it/s]

The experiment with the OpenAI model has yielded the best results. But, be aware! As we can see, there is a cost involved since we are using an API, and it needs to be paid for.

Another crucial piece of information is that we can view performance data for the models. This data could also be useful for minimally evaluating our inference server.